# Market Value Prediction — Exploratory Analysis
## FIFA 20 Dataset | Position-Specific PyTorch Models

Run `01_data_preparation.py` before opening this notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import RAW_CSV, PROC_DIR, POS_KEYS, POSITION_COLORS

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Shape: {df.shape}')
df[['short_name','age','overall','potential','value_eur','player_positions']].head(10)

## 2. Market Value Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
ax1.hist(df['value_eur'].dropna() / 1e6, bins=50, color='steelblue', edgecolor='white')
ax1.set_title('Market Value Distribution (EUR millions)')
ax1.set_xlabel('Value (€M)')

# Log distribution
ax2.hist(np.log1p(df['value_eur'].dropna()), bins=50, color='steelblue', edgecolor='white')
ax2.set_title('Log-transformed Market Value')
ax2.set_xlabel('log(Value EUR)')

plt.tight_layout()
plt.show()
print('The log transform converts the skewed distribution to near-normal — much better for regression!')

## 3. Correlation with Key Features

In [ ]:
corr_features = ['age','overall','potential','international_reputation',
                 'pace','shooting','passing','dribbling','defending','physic']
corr_df = df[corr_features + ['value_eur']].dropna()
corr_df['log_value'] = np.log1p(corr_df['value_eur'])

corr = corr_df[corr_features + ['log_value']].corr()['log_value'].drop('log_value').sort_values(ascending=False)

plt.figure(figsize=(10, 5))
corr.plot(kind='bar', color=['#2ECC71' if v > 0 else '#E74C3C' for v in corr])
plt.title('Feature Correlation with log(Market Value)')
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Correlation')
plt.tight_layout()
plt.show()

## 4. Value by Position Group

In [ ]:
# Load all processed position files
dfs = {}
for group, key in POS_KEYS.items():
    path = os.path.join(PROC_DIR, f'{key}_data.csv')
    if os.path.exists(path):
        dfs[group] = pd.read_csv(path)

fig, ax = plt.subplots(figsize=(10, 5))
for group, df_pos in dfs.items():
    ax.hist(np.log1p(df_pos['value_eur']), bins=40, alpha=0.5,
            label=f'{group} (n={len(df_pos):,})',
            color=POSITION_COLORS.get(group, 'gray'))

ax.set_title('Market Value Distribution by Position Group')
ax.set_xlabel('log(Value EUR)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()